# 02 — Architecture experiments

Define a network in a cell, train it with the **same pipeline as `ipe-train`** (frozen split,
target scaling, standardised MSE loss, `best.pt` / `last.pt` checkpoints, validation metrics),
and compare runs.

**Contract for a custom network** — a plain `nn.Module`:

| | shape | content |
|---|---|---|
| input `x` | `(B, 5, 480)` | `[ΔT / std, heater_on, t / L, cumsum(heater_on) / L, cumsum(ΔT / std) / L]` (1 step = 1 min) |
| output | `(B, 5)` | standardised `[HLC_mit, HLC_ext, log τ_b, log τ_n, log τ_inf]` |

With `tcn_extra_features=False` the input has only the first 2 channels. `HLC` is always computed
as `HLC_mit + HLC_ext`, so the additivity constraint holds by construction. Scaling, decoding to
physical units and the loss are handled by the pipeline.

`X.train(name, Net, epochs=..., **settings)` accepts any field of `DataConfig`, `ModelConfig` or
`TrainConfig` (`max_train`, `lr`, `batch_size`, `amp`, `weight_decay`, `temperature_noise_std`, ...).
Each run is saved in `outputs/runs/<name>/`; the network's source is saved next to it
(`network.py.txt`).

In [1]:
import os
from pathlib import Path

import pandas as pd
import torch
from torch import nn

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)  # the run configuration uses paths relative to the project root

from ipe_prediction.data import TARGET_NAMES
from ipe_prediction.training import experiment as X
from ipe_prediction.training.models import AttentionPooling, ResidualTCNBlock

pd.set_option("display.precision", 3)

# Budget shared by the runs of this notebook (raise for real comparisons).
EPOCHS = 100
MAX_TRAIN = None  # None = full training split (800k, ~1 min / epoch for the TCN)
OVERWRITE = True     # re-running a training cell restarts that run from scratch

tensors, split = X.load_data()  # loaded once, reused by every run (~2.4 GB RAM)
print({k: len(v) for k, v in split.items()}, "| input channels:", X.N_INPUTS, "| outputs:", X.N_OUTPUTS)

{'train': 800002, 'val': 99999, 'test': 99999} | input channels: 5 | outputs: 5


<module 'ipe_prediction.training.experiment' from 'C:\\Users\\ayman\\Projects\\IPE_prediction\\src\\ipe_prediction\\training\\experiment.py'>

## 1. Reference: built-in TCN + attention pooling

Built-in architectures are selected with `arch=` (`"tcn_attn"` or `"cnn_gru"`) and configured
with the `ModelConfig` fields (`tcn_channels`, `tcn_dilations`, `gru_hidden`, ...).

In [7]:
#res_ref = X.train("nb_tcn_attn", arch="tcn_attn", epochs=EPOCHS, max_train=MAX_TRAIN, overwrite=OVERWRITE)
#pd.DataFrame(res_ref["val_family"]).T  # per-family validation metrics of best.pt

## 2. Custom architecture: plain CNN + global average pooling

A deliberately simple model: strided convolutions then an average over time.

In [8]:
class SimpleCNN(nn.Module):
    def __init__(self, n_in: int = X.N_INPUTS, width: int = 64) -> None:
        super().__init__()
        layers, c = [], n_in
        for c_out in (32, width, width, width):
            layers += [nn.Conv1d(c, c_out, kernel_size=5, stride=2, padding=2), nn.BatchNorm1d(c_out), nn.GELU()]
            c = c_out
        self.conv = nn.Sequential(*layers)  # 480 -> 30 steps
        self.head = nn.Sequential(nn.Linear(c, 64), nn.GELU(), nn.Linear(64, X.N_OUTPUTS))

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # (B, F, L) -> (B, 5)
        return self.head(self.conv(x).mean(dim=2))

In [9]:
res_cnn = X.train("nb_simple_cnn", SimpleCNN, epochs=EPOCHS, max_train=MAX_TRAIN, overwrite=OVERWRITE)

2026-11-15 23:22:36,532 data ready in 0s | train 800002 | val 10000
2026-11-15 23:22:37,982 model custom: 57,157 parameters | delta_t std 4.789 K | batch 512 | amp True
2026-11-15 23:22:53,479 epoch 001 | train 0.8745 | val 0.8652 | val MAPE % HLC=9.10 HLC_mit=16.20 HLC_ext=12.06 tau_b=15.86 tau_n=13.19 tau_inf=18.86 | 15s | lr 1e-03
2026-11-15 23:23:08,638 epoch 002 | train 0.8583 | val 0.9023 | val MAPE % HLC=9.29 HLC_mit=16.02 HLC_ext=11.93 tau_b=15.95 tau_n=13.22 tau_inf=18.55 | 15s | lr 1e-03
2026-11-15 23:23:23,001 epoch 003 | train 0.8461 | val 0.8446 | val MAPE % HLC=9.00 HLC_mit=16.08 HLC_ext=11.72 tau_b=15.52 tau_n=12.93 tau_inf=18.51 | 14s | lr 1e-03
2026-11-15 23:23:37,354 epoch 004 | train 0.8353 | val 0.8429 | val MAPE % HLC=9.09 HLC_mit=16.40 HLC_ext=11.87 tau_b=15.43 tau_n=12.99 tau_inf=18.51 | 14s | lr 1e-03
2026-11-15 23:23:52,842 epoch 005 | train 0.8280 | val 0.8391 | val MAPE % HLC=8.88 HLC_mit=15.92 HLC_ext=11.44 tau_b=15.39 tau_n=12.76 tau_inf=18.29 | 15s | lr 1e

KeyboardInterrupt: 

## 3. Custom architecture built from the package blocks

`ResidualTCNBlock` and `AttentionPooling` (from `ipe_prediction.training.models`) can be reused
to try variants, e.g. a wider TCN with more attention heads. Constructor arguments are passed
with a `lambda`.

In [ ]:
class WideTCN(nn.Module):
    def __init__(self, n_in: int = X.N_INPUTS, channels: int = 96, heads: int = 8,
                 dilations: tuple[int, ...] = (1, 2, 4, 8, 16, 32, 64), dropout: float = 0.1) -> None:
        super().__init__()
        self.stem = nn.Conv1d(n_in, channels, kernel_size=5, stride=2, padding=2)  # 480 -> 240
        self.blocks = nn.Sequential(*(ResidualTCNBlock(channels, 3, d, dropout) for d in dilations))
        self.norm = nn.Sequential(nn.BatchNorm1d(channels), nn.GELU())
        self.pool = AttentionPooling(channels, heads)
        self.head = nn.Sequential(nn.Linear(channels, 128), nn.GELU(), nn.Linear(128, X.N_OUTPUTS))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.pool(self.norm(self.blocks(self.stem(x)))))

In [ ]:
res_wide = X.train("nb_wide_tcn", lambda: WideTCN(channels=96, heads=8), epochs=EPOCHS,
                   max_train=MAX_TRAIN, lr=5e-4, overwrite=OVERWRITE)

## 4. Comparison

Best validation epoch of each run (validation loss: 1.0 = predicting the training mean) and the
learning curves.

In [ ]:
RUNS = ["nb_tcn_attn", "nb_simple_cnn", "nb_wide_tcn"]
display(X.compare(RUNS))
X.plot_histories(RUNS);

## 5. Reload a trained model and inspect predictions

A custom run is rebuilt from the same factory; built-in runs need only the name. To continue a
run, call `X.train(name, Net, epochs=<new total>, resume=True)`.

In [ ]:
model = X.load_model("nb_simple_cnn", SimpleCNN)
idx = split["val"][:5]
pred = model.predict(tensors.delta_t[idx], tensors.heater_on[idx].float())
true = tensors.targets[idx]
pd.concat(
    {n: pd.DataFrame({"true": true[:, j], "pred": pred[:, j]}) for j, n in enumerate(TARGET_NAMES)},
    axis=1,
)